# Notebook 03: Statistical Inference & Hypothesis Testing

## Credit Card Customer Churn & Segmentation
**Objective:** Formulate and statistically test core research questions (RQ1–RQ10) and hypotheses (H1–H7) using Chi-Square tests of independence, Welch's t-tests, Mann-Whitney U tests, 95% confidence intervals, and effect size calculations.

> **Methodological Standard:** We report test statistics, p-values, effect sizes (Cohen's d, Cramér's V), and confidence intervals. We strictly avoid causal assertions, framing findings as statistically verified associations.


In [ ]:
import pandas as pd
import numpy as np
import json
from scipy import stats

from src.data.load_data import load_raw_data
from src.features.build_features import add_engineered_features
from src.data.statistical_analysis import run_categorical_tests, run_numerical_tests
from src.utils.helpers import TARGET_COLUMN

df = load_raw_data()
df = add_engineered_features(df)
print(f"Loaded dataset with {df.shape[1]} features.")

## 1. Categorical Associations (Chi-Square Tests of Independence)
We test whether churn is independent of demographic and card tiers:
- Null Hypothesis $H_0$: Churn status is independent of category $X$.
- Significance level $\alpha = 0.05$.
- Effect size measured via Cramér's V.

In [ ]:
cat_tests = run_categorical_tests(df)
cat_summary = pd.DataFrame([
    {
        "Feature": t["feature"],
        "Chi2 Stat": t["chi2_statistic"],
        "p-value": f"{t['p_value']:.4e}",
        "Cramér's V": t["cramers_v"],
        "Effect Size": t["effect_size_label"],
        "Significant?": "Yes" if t["is_significant"] else "No"
    }
    for t in cat_tests
])
cat_summary

## 2. Numerical Feature Testing (Parametric & Non-Parametric)
For continuous variables, we evaluate differences between Churned and Retained customers using:
1. **Welch's t-test** (robust to unequal variances)
2. **Mann-Whitney U test** (non-parametric rank sum test)
3. **95% Confidence Interval for Difference in Means**
4. **Cohen's d** (standardized effect size)

In [ ]:
num_tests = run_numerical_tests(df)
num_summary = pd.DataFrame([
    {
        "Feature": t["feature"],
        "Mean Retained": t["mean_retained"],
        "Mean Churned": t["mean_churned"],
        "Diff Mean": t["diff_mean"],
        "95% CI Diff": f"[{t['ci_95_diff'][0]}, {t['ci_95_diff'][1]}]",
        "Mann-Whitney p": f"{t['mann_whitney_p_value']:.2e}",
        "Cohen's d": t["cohens_d"],
        "Effect Label": t["effect_size_label"],
    }
    for t in num_tests
])
num_summary

## 3. Formal Testing of Initial Hypotheses (H1–H7)

| Hypothesis | Test Description | Finding | Conclusion |
|---|---|---|---|
| **H1: Inactivity** | Customers with more inactive months churn more | Mann-Whitney $p < 10^{-20}$, Cohen's $d = +0.33$ | **Supported** (Positive association) |
| **H2: Transaction Count** | Customers with fewer transactions churn more | Mann-Whitney $p < 10^{-250}$, Cohen's $d = -1.09$ | **Supported** (Large negative effect) |
| **H3: Transaction Momentum** | Declining transaction count Q4/Q1 associates with churn | Mann-Whitney $p < 10^{-200}$, Cohen's $d = -0.83$ | **Supported** (Large negative effect) |
| **H4: Relationship Depth** | Customers with fewer products churn more | Mann-Whitney $p < 10^{-50}$, Cohen's $d = -0.39$ | **Supported** (Moderate negative effect) |
| **H5: Bank Contacts** | Frequent customer contact associates with churn | Mann-Whitney $p < 10^{-80}$, Cohen's $d = +0.47$ | **Supported** (Moderate positive effect) |
| **H6: Feature Priority** | Behavioral variables provide stronger signal than demographics | Cramér's V for demographics $< 0.04$ vs Cohen's d for transactions $> 1.0$ | **Supported** |
| **H7: Segments** | Natural segments show distinct churn risk | Tested in Notebook 06 | Deferred to Clustering |
